# World Cup 2026 — Modelo predictivo

Notebook autocontenido (sin dependencias del repo) que entrena un modelo de **strength** por selección combinando:

- Ranking FIFA
- Factores socioeconómicos (GDP per cápita, población, IDH)
- Factores geográficos (continente, distancia a sede, ventaja de anfitrión)
- Datos históricos de los últimos 10 mundiales

Luego simula 10 000 mundiales con Monte Carlo y devuelve P(campeón) por selección.

## 1. Instalación

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib

## 2. Descarga de datasets desde GitHub

Si usás este notebook desde el repo, ajustá `BASE_URL` al raw content de tu branch.

In [ ]:
import urllib.request
from pathlib import Path

BASE_URL = "https://raw.githubusercontent.com/diegoarpe/cca-quest/claude/world-cup-prediction-model-eLGVE/world-cup-2026/data"
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)

for fname in ["historical_worldcups.csv", "qualified_2026.csv", "hosts_2026.csv"]:
    urllib.request.urlretrieve(f"{BASE_URL}/{fname}", DATA_DIR / fname)
    print(f"✓ {fname}")

## 3. Feature engineering

In [ ]:
import math
import numpy as np
import pandas as pd

ROUND_TO_SCORE = {"Group":1,"Round16":2,"Quarterfinal":3,"Semifinal":4,"Final":5,"Winner":6}

HOST_COORDS = {
    "Mexico":(19.43,-99.13),"Italy":(41.90,12.50),"USA":(38.90,-77.04),
    "France":(48.86,2.35),"South Korea":(37.57,126.98),"Germany":(52.52,13.41),
    "South Africa":(-25.75,28.19),"Brazil":(-15.78,-47.93),
    "Russia":(55.75,37.62),"Qatar":(25.29,51.53),
}

TEAM_COORDS = {
    "Argentina":(-34.6,-58.4),"Brazil":(-15.8,-47.9),"Uruguay":(-34.9,-56.2),
    "Paraguay":(-25.3,-57.6),"Chile":(-33.4,-70.6),"Colombia":(4.7,-74.1),
    "Ecuador":(-0.2,-78.5),"Peru":(-12.0,-77.0),"Bolivia":(-16.5,-68.1),
    "Venezuela":(10.5,-66.9),"United States":(38.9,-77.0),"Mexico":(19.4,-99.1),
    "Canada":(45.4,-75.7),"Costa Rica":(9.9,-84.1),"Honduras":(14.1,-87.2),
    "Panama":(8.9,-79.5),"Jamaica":(18.1,-76.8),"France":(48.8,2.3),
    "Germany":(52.5,13.4),"West Germany":(50.1,8.7),"Spain":(40.4,-3.7),
    "Italy":(41.9,12.5),"England":(51.5,-0.1),"Netherlands":(52.4,4.9),
    "Portugal":(38.7,-9.1),"Belgium":(50.8,4.4),"Croatia":(45.8,15.9),
    "Switzerland":(46.9,7.4),"Denmark":(55.7,12.6),"Sweden":(59.3,18.1),
    "Norway":(59.9,10.7),"Poland":(52.2,21.0),"Austria":(48.2,16.4),
    "Czechoslovakia":(50.1,14.4),"Yugoslavia":(44.8,20.5),"Serbia":(44.8,20.5),
    "Bulgaria":(42.7,23.3),"Romania":(44.4,26.1),"Greece":(37.98,23.73),
    "Turkey":(39.9,32.9),"Ireland":(53.3,-6.3),"Republic of Ireland":(53.3,-6.3),
    "Ukraine":(50.5,30.5),"Russia":(55.8,37.6),"Soviet Union":(55.8,37.6),
    "Slovakia":(48.1,17.1),"Scotland":(55.9,-3.2),"Morocco":(34.0,-6.8),
    "Egypt":(30.0,31.2),"Algeria":(36.8,3.1),"Tunisia":(36.8,10.2),
    "Senegal":(14.7,-17.5),"Nigeria":(9.1,7.5),"Cameroon":(3.9,11.5),
    "Ghana":(5.6,-0.2),"Ivory Coast":(7.5,-5.5),"Japan":(35.7,139.7),
    "South Korea":(37.5,127.0),"Iran":(35.7,51.4),"Saudi Arabia":(24.7,46.7),
    "Qatar":(25.3,51.5),"Iraq":(33.3,44.4),"Australia":(-35.3,149.1),
    "Uzbekistan":(41.3,69.2),"Jordan":(31.9,35.9),"New Zealand":(-41.3,174.8),
}

CONF_TO_CONTINENT = {"CONMEBOL":"South America","CONCACAF":"North America",
                     "UEFA":"Europe","CAF":"Africa","AFC":"Asia","OFC":"Oceania"}

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2-lat1); dlambda = math.radians(lon2-lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2*r*math.asin(math.sqrt(a))

def dist_to_host(team, host):
    t = TEAM_COORDS.get(team); h = HOST_COORDS.get(host)
    if t is None or h is None: return 8000.0
    return haversine_km(t[0], t[1], h[0], h[1])

def engineer_historical(df):
    out = df.copy()
    out["is_host"] = (out["team"] == out["host"]).astype(int)
    out["same_continent_host"] = (out["continent"] == out["host_continent"]).astype(int)
    out["distance_to_host_km"] = out.apply(lambda r: dist_to_host(r["team"], r["host"]), axis=1)
    out["log_gdp_per_capita"] = np.log1p(out["gdp_per_capita_usd"])
    out["log_population"] = np.log1p(out["population_millions"])
    out["log_distance_to_host"] = np.log1p(out["distance_to_host_km"])
    out["inv_fifa_rank"] = 1.0 / out["fifa_rank_at_start"].clip(lower=1)
    out["log_fifa_rank"] = np.log1p(out["fifa_rank_at_start"])
    out["elite_pedigree"] = out["prior_titles"]*2 + out["prior_participations"]/4.0
    out["is_uefa"] = (out["continent"] == "UEFA").astype(int)
    out["is_conmebol"] = (out["continent"] == "CONMEBOL").astype(int)
    return out

def engineer_2026(qualified, hosts):
    out = qualified.copy()
    out["continent_bucket"] = out["confederation"].map(CONF_TO_CONTINENT).fillna("Europe")
    out["host_continent"] = "CONCACAF"
    out["same_continent_host"] = (out["confederation"] == "CONCACAF").astype(int)
    out["is_host"] = out["host"].astype(int)
    def nearest(row):
        return min(haversine_km(row["latitude"], row["longitude"], h["latitude"], h["longitude"]) for _,h in hosts.iterrows())
    out["distance_to_host_km"] = out.apply(nearest, axis=1)
    out["log_gdp_per_capita"] = np.log1p(out["gdp_per_capita_usd"])
    out["log_population"] = np.log1p(out["population_millions"])
    out["log_distance_to_host"] = np.log1p(out["distance_to_host_km"])
    out["inv_fifa_rank"] = 1.0 / out["fifa_rank"].clip(lower=1)
    out["log_fifa_rank"] = np.log1p(out["fifa_rank"])
    out["fifa_rank_at_start"] = out["fifa_rank"]
    out["elite_pedigree"] = out["prior_titles"]*2 + out["prior_participations"]/4.0
    out["is_uefa"] = (out["confederation"] == "UEFA").astype(int)
    out["is_conmebol"] = (out["confederation"] == "CONMEBOL").astype(int)
    return out

FEATURE_COLUMNS = ["inv_fifa_rank","log_fifa_rank","log_gdp_per_capita","log_population",
                   "hdi","elite_pedigree","prior_titles","prior_participations",
                   "is_host","same_continent_host","log_distance_to_host",
                   "is_uefa","is_conmebol"]

hist = pd.read_csv(DATA_DIR/"historical_worldcups.csv")
hist["round_score"] = hist["final_round"].map(ROUND_TO_SCORE)
hist = engineer_historical(hist)
qual = pd.read_csv(DATA_DIR/"qualified_2026.csv")
hosts = pd.read_csv(DATA_DIR/"hosts_2026.csv")
df_2026 = engineer_2026(qual, hosts)
print(f"Historical: {len(hist)} rows | 2026: {len(df_2026)} teams")

## 4. Entrenamiento (Ridge + GBM blend)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

def make_ridge():
    return Pipeline([("s", StandardScaler()), ("r", Ridge(alpha=3.0, random_state=42))])

def make_gbm():
    return GradientBoostingRegressor(n_estimators=500, learning_rate=0.03, max_depth=2,
                                     subsample=0.8, min_samples_leaf=5, random_state=42)

X = hist[FEATURE_COLUMNS].to_numpy(); y = hist["round_score"].to_numpy(); g = hist["year"].to_numpy()

fold_errs = []
for tr, te in GroupKFold(n_splits=5).split(X, y, g):
    r = make_ridge().fit(X[tr], y[tr]); b = make_gbm().fit(X[tr], y[tr])
    pred = 0.6*r.predict(X[te]) + 0.4*b.predict(X[te])
    fold_errs.append(mean_absolute_error(y[te], pred))
cv_mae = float(np.mean(fold_errs))
print(f"CV MAE (round_score 1-6): {cv_mae:.3f}")

ridge_final = make_ridge().fit(X, y); gbm_final = make_gbm().fit(X, y)

ridge_coefs = pd.Series(ridge_final.named_steps["r"].coef_, index=FEATURE_COLUMNS).abs()
ridge_imp = ridge_coefs / ridge_coefs.sum()
gbm_imp = pd.Series(gbm_final.feature_importances_, index=FEATURE_COLUMNS)
importance = (0.6*ridge_imp + 0.4*gbm_imp).sort_values(ascending=False)
print("\nFeature importance:"); print(importance.to_string())

## 5. Predicción de strength 2026

In [ ]:
X26 = df_2026[FEATURE_COLUMNS].to_numpy()
ml = 0.6*ridge_final.predict(X26) + 0.4*gbm_final.predict(X26)
rank = df_2026["fifa_rank"].to_numpy()
rank_prior = 5.5 - 3.5 * (np.log1p(rank) / np.log1p(48))
strength = 0.55*ml + 0.45*rank_prior

teams = df_2026.copy(); teams["strength"] = strength
scored = teams[["team","confederation","fifa_rank","strength"]].sort_values("strength", ascending=False).reset_index(drop=True)
print("Top 15 por strength predicho:"); scored.head(15)

## 6. Monte Carlo del torneo (10k simulaciones)

In [ ]:
def strength_to_xg(s): return max(0.3, 0.45*s + 0.5)

def sim_match(rng, sa, sb, ko=False):
    ma = max(0.2, strength_to_xg(sa) * (1 + 0.12*(sa-sb)))
    mb = max(0.2, strength_to_xg(sb) * (1 + 0.12*(sb-sa)))
    ga, gb = rng.poisson(ma), rng.poisson(mb)
    if ga>gb: return ga, gb, 0
    if gb>ga: return ga, gb, 1
    if not ko: return ga, gb, -1
    p = float(np.clip(0.5 + 0.07*(sa-sb), 0.15, 0.85))
    return ga, gb, (0 if rng.random()<p else 1)

def assign_groups(rng, teams):
    t = teams.sort_values("fifa_rank").reset_index(drop=True)
    hosts_t = t[t["host"]==1]["team"].tolist()
    non = t[t["host"]==0]["team"].tolist()
    pot1 = hosts_t + non[:12-len(hosts_t)]
    rest = [x for x in t["team"].tolist() if x not in pot1]
    pot2, pot3, pot4 = rest[:12], rest[12:24], rest[24:36]
    for p in (pot2, pot3, pot4): rng.shuffle(p)
    groups = [[pot1[i]] for i in range(12)]
    for p in (pot2, pot3, pot4):
        for i in range(12): groups[i].append(p[i])
    return groups

def sim_group(rng, group, strength):
    s = {t:{"pts":0,"gf":0,"ga":0} for t in group}
    for i in range(len(group)):
        for j in range(i+1, len(group)):
            a,b = group[i], group[j]
            ga, gb, w = sim_match(rng, strength[a], strength[b])
            s[a]["gf"]+=ga; s[a]["ga"]+=gb; s[b]["gf"]+=gb; s[b]["ga"]+=ga
            if w==0: s[a]["pts"]+=3
            elif w==1: s[b]["pts"]+=3
            else: s[a]["pts"]+=1; s[b]["pts"]+=1
    return sorted(s.items(), key=lambda kv:(kv[1]["pts"], kv[1]["gf"]-kv[1]["ga"], kv[1]["gf"]), reverse=True)

def ko(rng, pairs, strength):
    out = []
    for a,b in pairs:
        _,_,w = sim_match(rng, strength[a], strength[b], ko=True)
        out.append(a if w==0 else b)
    return out

def simulate_tournament(rng, teams, tally):
    strength = dict(zip(teams["team"], teams["strength"]))
    groups = assign_groups(rng, teams)
    advanced = []; thirds = []
    for g in groups:
        st = sim_group(rng, g, strength)
        advanced += [st[0][0], st[1][0]]
        thirds.append((st[2][0], st[2][1]["pts"], st[2][1]["gf"]-st[2][1]["ga"], st[2][1]["gf"]))
    thirds.sort(key=lambda x:(x[1],x[2],x[3]), reverse=True)
    advanced += [t[0] for t in thirds[:8]]
    rng.shuffle(advanced)
    r16 = ko(rng, [(advanced[i], advanced[i+1]) for i in range(0,32,2)], strength)
    qf = ko(rng, [(r16[i], r16[i+1]) for i in range(0,16,2)], strength)
    tally["qf"].extend(qf)
    sf = ko(rng, [(qf[i], qf[i+1]) for i in range(0,8,2)], strength)
    tally["sf"].extend(sf)
    fn = ko(rng, [(sf[0],sf[1]), (sf[2],sf[3])], strength)
    tally["fn"].extend(fn)
    champ = ko(rng, [(fn[0], fn[1])], strength)[0]
    tally["wn"].append(champ)

N = 10_000
rng = np.random.default_rng(2026)
tally = {"wn":[],"fn":[],"sf":[],"qf":[]}
for _ in range(N): simulate_tournament(rng, teams, tally)

def counts(xs): return pd.Series(xs).value_counts().to_dict()
wn, fn, sf, qf = map(counts, (tally["wn"], tally["fn"], tally["sf"], tally["qf"]))
probs = pd.DataFrame([{
    "team":t,
    "P(winner)": wn.get(t,0)/N,
    "P(final)": fn.get(t,0)/N,
    "P(semifinal)": sf.get(t,0)/N,
    "P(quarterfinal)": qf.get(t,0)/N,
} for t in teams["team"]]).sort_values("P(winner)", ascending=False).reset_index(drop=True)

print(f"Candidato más probable: {probs.iloc[0]['team']} (P={probs.iloc[0]['P(winner)']:.1%})")
probs.head(20)

## 7. Visualización

In [ ]:
import matplotlib.pyplot as plt

top = probs.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(10,8))
ax.barh(top["team"], top["P(winner)"]*100, color="#1f77b4")
ax.set_xlabel("P(campeón) %")
ax.set_title("Mundial 2026 — Top 20 candidatos (Monte Carlo, N=10 000)")
for i,v in enumerate(top["P(winner)"]*100):
    ax.text(v+0.2, i, f"{v:.1f}%", va="center", fontsize=9)
plt.tight_layout(); plt.show()

imp = importance.sort_values()
fig, ax = plt.subplots(figsize=(9,6))
ax.barh(imp.index, imp.values, color="#d95f02")
ax.set_xlabel("Importancia (blend Ridge + GBM)")
ax.set_title("Feature importance")
plt.tight_layout(); plt.show()